In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")


In [ ]:

# Locate the project root
if Path("data/cleaned_dataset.csv").exists():
    PROJECT_ROOT = Path(".")
elif Path("../data/cleaned_dataset.csv").exists():
    PROJECT_ROOT = Path("..")
else:
    raise FileNotFoundError(
        "data/cleaned_dataset.csv not found. Complete Day 2 first."
    )

df = pd.read_csv(PROJECT_ROOT / "data" / "cleaned_dataset.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())


In [ ]:

df["question_char_count"] = (
    df["test_question"].astype(str).str.len()
)

df["prompt_char_count"] = (
    df["system_prompt"].astype(str).str.len()
)

df["question_prompt_length_ratio"] = (
    df["question_char_count"] /
    df["prompt_char_count"].replace(0, np.nan)
)

df["question_word_count"] = (
    df["test_question"].astype(str).str.split().str.len()
)

print("Engineered features created!")
display(df[[
    "question_word_count",
    "question_prompt_length_ratio"
]].head())



In [ ]:

# Select only the two meaningful engineered features
features = [
    "question_word_count",
    "question_prompt_length_ratio"
]

X = df[features].copy()

print("Features selected for scaling:")
print(X.head())

print("\nFeature statistics before scaling:")
display(X.describe())

print("\nFeature variance:")
display(X.var())


In [ ]:

# Check for missing and infinite values
assert X.notnull().all().all(), "Missing values found!"
assert np.isfinite(X.to_numpy()).all(), "Infinite values found!"

# Create and fit the scaler
scaler = StandardScaler()

X_scaled_array = scaler.fit_transform(X)

# Convert scaled values back to a DataFrame
X_scaled = pd.DataFrame(
    X_scaled_array,
    columns=features,
    index=X.index
)

print("Scaling completed successfully!")

print("\nBefore scaling:")
display(X.head())

print("\nAfter scaling:")
display(X_scaled.head())


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for i, feature in enumerate(features):
    sns.histplot(
        X[feature],
        bins=8,
        kde=True,
        ax=axes[i, 0]
    )
    axes[i, 0].set_title(f"{feature} - Before Scaling")

    sns.histplot(
        X_scaled[feature],
        bins=8,
        kde=True,
        ax=axes[i, 1]
    )
    axes[i, 1].set_title(f"{feature} - After Scaling")

plt.tight_layout()
plt.show()


In [ ]:

comparison = pd.DataFrame({
    "Original Mean": X.mean(),
    "Scaled Mean": X_scaled.mean(),
    "Original Std": X.std(ddof=0),
    "Scaled Std": X_scaled.std(ddof=0)
})

display(comparison)


In [ ]:

# Create models directory
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Save the fitted scaler
scaler_path = models_dir / "scaler.pkl"

joblib.dump(scaler, scaler_path)

print("Scaler saved successfully!")
print("Location:", scaler_path)


In [ ]:

# Load the saved scaler
loaded_scaler = joblib.load(scaler_path)

# Test that the loaded scaler produces the same results
test_scaled = loaded_scaler.transform(X)

assert np.allclose(test_scaled, X_scaled_array)

assert np.allclose(X_scaled.mean().values, 0, atol=1e-7)
assert np.allclose(X_scaled.std(ddof=0).values, 1, atol=1e-7)

print("SUCCESS: Scaler saved and verified!")
print("Features:", loaded_scaler.feature_names_in_.tolist())
print("No missing values:", X_scaled.notnull().all().all())


# Week 2 – Day 4: Feature Scaling

## Objective
Apply feature scaling to numerical features to make them suitable for machine learning algorithms that are sensitive to feature magnitudes.

## Selected Features
1. `question_word_count`
2. `question_prompt_length_ratio`

The `id` column was excluded because it is an identifier rather than a meaningful predictive feature.

## Scaling Method: StandardScaler

StandardScaler was selected because the two engineered features have different numerical ranges. It transforms features to approximately zero mean and unit variance.

### Why StandardScaler?
- Useful for gradient-based models such as logistic regression and neural networks.
- Helps prevent features with larger numerical ranges from dominating optimization.
- Does not constrain values to the [0, 1] range.

MinMaxScaler was not selected because bounded inputs are not currently a requirement.

## Before and After Comparison
Histograms and descriptive statistics were compared before and after scaling.

## Saved Model Artifact
The fitted scaler was saved at `models/scaler.pkl` for reuse in the serving pipeline.

## Conclusion
Two engineered numeric features were standardized using StandardScaler. The fitted scaler was saved and reloaded successfully. The same fitted scaler and feature ordering must be used when processing future inputs.